# Mouse view from the log — one frame, both icon types on screen

Reconstruct **what the mouse saw on its monitor** at a chosen moment of a `banish_multiplier` session,
straight from the log, and save it as a PNG. For a trial (picked by **frame** or **timestamp**) this:

1. reads the **avatar** position/heading and **every icon** (reward / banishment / escape) on the board;
2. finds a frame where **both icon types are on screen at the same time** (the interesting case);
3. renders the mouse-view PNG (`reconstruct_view.MouseView.render`) with the on-screen icons ringed;
4. plots the **world-space map** — avatar path, icon locations, and the viewport box at that frame.

> **Dependency.** The renderer needs `df_trials_clean.pkl` (the authority on which world/icons are on
> screen — the log does not record the NORMAL↔SHADOW-REALM swap) and the game art in
> `<session>/task/assets_game/`. It is specific to the **banishment** protocol.

In [ ]:
import sys, json
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt

# locate common/ + the banish task (this notebook lives in optic_flow/presentations/)
_HERE = Path.cwd()
def _find_up(marker, rel):
    for base in (_HERE, _HERE.parent, _HERE.parent.parent, _HERE.parent.parent.parent):
        if (base / rel / marker).exists():
            return base / rel
    return None
_COMMON = _find_up('viewport.py', 'common')
_TASK   = _find_up('reconstruct_view.py', 'task_banish_multiplier')
assert _COMMON and _TASK, f'cannot locate common/ + task_banish_multiplier/ from {_HERE}'
sys.path.insert(0, str(_COMMON)); sys.path.insert(0, str(_TASK))
import reconstruct_view as rv
print('common ->', _COMMON, '| task ->', _TASK)

In [ ]:
# ── CONFIG: point at ONE banishment session (same layout as notebook 0) ────────
MAIN_DIR = '/path/to/MAIN_DIR'    # <-- server root that holds the animal folders
MOUSE_ID = 'MOUSE_ID'             # <-- animal folder name
date     = ''                     # <-- session sub-folder (recording date/time); '' = none
VISION   = 'rgb'                  # 'rgb' = as a human sees it | 'mouse' = red dropped (what the mouse sees)

SESSION_DIR = (Path(MAIN_DIR) / MOUSE_ID / date).resolve()
assert SESSION_DIR.exists(), f'session folder does not exist: {SESSION_DIR}  (set MAIN_DIR / MOUSE_ID / date)'
PRES_DIR = SESSION_DIR / 'presentations'          # PNGs go here by default

try:
    MV = rv.MouseView(str(SESSION_DIR), vision=VISION)
except SystemExit as ex:
    raise FileNotFoundError(f'{ex}\n-> mouse_view needs df_trials_clean.pkl + task/assets_game/ '
                            '(and a banishment session). Run the trial pipeline first.')
print(f'view {MV.view_w:.0f} x {MV.view_h:.0f} world units (scale {MV.scale}), '
      f'world {MV.W:.0f}x{MV.H:.0f}, icons {MV.icon_w:.0f} wu, vision={MV.vision}')
print('trials:', len(MV.df), '| frames:', int(MV.sess['n_frames']))

## 1 — Read the trial: avatar + every icon

`visible_icons(f)` returns the avatar (world x/y + heading) and, for that frame's board, **every icon
with its world position and whether its centre is inside the viewport** (the same centre-in-box test
the performance code uses). Reward = green, banishment = blue, escape ring = teal.
Pick a frame directly, from a timestamp (`frame_at_ms`), or from a trial (`trial_window`).

In [ ]:
REWARD = {'single_reward', 'double_reward', 'money'}; BANISH = {'banish'}; ESCAPE = {'unbanish'}
def _cls(e): return ('reward' if e in REWARD else 'banish' if e in BANISH else
                     'escape' if e in ESCAPE else e)
COLOR = {'reward': (90, 230, 90), 'banish': (240, 150, 60), 'escape': (200, 200, 90)}   # BGR

def frame_at_ms(ms):     return int(np.argmin(np.abs(MV.frame_ms - float(ms))))
def trial_window(n):
    r = MV.df[MV.df.trial == n].iloc[0]; return int(r['start_frame']), int(r['end_frame'])

def visible_icons(f):
    '''(avatar_x, avatar_y, theta, [icon dicts]). Each icon: effect, cls, world x/y, on(screen), margin
    (world units its centre is inside the viewport edge; negative = off screen).'''
    ax, ay, th, dx, dy = MV.avatar_at(int(f)); vw, vh = MV.view_w / 2, MV.view_h / 2
    ics = [dict(effect=i['effect'], cls=_cls(i['effect']), x=float(i['x']), y=float(i['y']),
                on=bool(abs(i['x'] - ax) <= vw and abs(i['y'] - ay) <= vh),
                margin=float(min(vw - abs(i['x'] - ax), vh - abs(i['y'] - ay))))
           for i in MV.icons_of[int(f)]]
    return ax, ay, th, ics

def icon_table(f):
    ax, ay, th, ics = visible_icons(f)
    print(f'frame {f}  t={MV.frame_ms[int(f)]:.0f} ms  trial {MV.trial_of[int(f)]}  '
          f'{MV.world_of[int(f)]}   avatar=({ax:.0f}, {ay:.0f})  theta={th:+.2f}')
    for i in ics:
        print(f"   {i['cls']:7s} {i['effect']:14s} ({i['x']:6.0f}, {i['y']:6.0f})  "
              f"{'ON screen' if i['on'] else 'off screen'}   margin {i['margin']:+.0f} wu")
    return ax, ay, th, ics

## 2 — Find a frame where BOTH icon types are on screen

Scans a trial (or a frame range, or the whole session) for frames where at least one **reward** AND one
**banishment** icon are simultaneously on screen, and returns the one where both sit most comfortably
inside the viewport (largest minimum margin). This is the "he could see the choice" frame.

In [ ]:
def both_visible(f):
    _, _, _, ics = visible_icons(f); t = {i['cls'] for i in ics if i['on']}
    return ('reward' in t) and ('banish' in t)

def find_both(trial=None, lo=None, hi=None, world='NORMAL'):
    '''Best frame with a reward AND a banishment on screen. Give a trial, a [lo,hi) range, or neither
    (scan every analyzable NORMAL trial and take the first that has such a frame).'''
    def _best(a, b):
        fs = [f for f in range(int(a), int(b)) if both_visible(f)]
        if not fs: return None
        def score(f):
            _, _, _, ics = visible_icons(f)
            m = [i['margin'] for i in ics if i['on'] and i['cls'] in ('reward', 'banish')]
            return min(m)
        return max(fs, key=score), len(fs)
    if trial is not None:
        a, b = trial_window(trial); r = _best(a, b)
        if r is None: raise RuntimeError(f'trial {trial}: no frame with both icon types on screen')
        print(f'trial {trial}: {r[1]} both-visible frames -> best frame {r[0]}'); return r[0]
    if lo is not None:
        r = _best(lo, hi or MV.sess['n_frames'])
        if r is None: raise RuntimeError('no both-visible frame in that range'); 
        print(f'range [{lo},{hi}): {r[1]} both-visible frames -> best frame {r[0]}'); return r[0]
    for _, row in MV.df.iterrows():                       # scan the trials
        if row['world'] != world or (world == 'NORMAL' and not row.get('analyze', True)): continue
        r = _best(row['start_frame'], row['end_frame'])
        if r: print(f"trial {int(row['trial'])}: {r[1]} both-visible frames -> best frame {r[0]}"); return r[0]
    raise RuntimeError('no trial had both icon types on screen at once')

F = find_both()          # or find_both(trial=5) / find_both(lo=1000, hi=2000)
icon_table(F);

## 3 — Render the mouse-view PNG

`reconstruct_view.MouseView.render(f)` rebuilds the exact monitor image (world texture + black border
where the world runs out + the on-screen icons + the green breathing avatar + heading arrow + the
game's global dim). `mouse_view_png` rings the on-screen icons, captions the frame, and saves it.
`vision='mouse'` drops the red channel (the terrain nearly vanishes, the icons stay bright — what the
mouse actually sees). Output like the other tools: default a PNG in `<session>/presentations/`;
`fmt='pdf'` / `save_path=` / `outdir=` redirect.

In [ ]:
def mouse_view_png(frame, annotate=True, vision=None, arrow=True, outdir=None, save_path=None,
                   fmt=None, show=True):
    '''Render + save the mouse view at `frame`. Rings the on-screen icons (reward=green, banishment=
    blue, escape=teal) and captions trial / frame / time / world.'''
    f = int(frame)
    if vision and vision != MV.vision: MV.vision = vision      # toggle rgb <-> mouse
    img = MV.render(f, show_arrow=arrow).copy()
    ax, ay, th, ics = visible_icons(f)
    if annotate:
        for i in ics:
            if not i['on']: continue
            sx, sy = MV._w2s(i['x'], i['y'], ax, ay); c = COLOR.get(i['cls'], (255, 255, 255))
            cv2.circle(img, (sx, sy), 28, c, 2, cv2.LINE_AA)
            cv2.putText(img, i['cls'], (sx - 26, sy - 34), cv2.FONT_HERSHEY_SIMPLEX, 0.5, c, 1, cv2.LINE_AA)
        cv2.putText(img, f'trial {MV.trial_of[f]}   frame {f}   t={MV.frame_ms[f]:.0f} ms   {MV.world_of[f]}',
                    (8, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (235, 235, 235), 1, cv2.LINE_AA)
    ext = (fmt or 'png').lstrip('.')
    if save_path:
        p = Path(save_path)
        if not p.suffix: p = p / f'mouse_view_f{f}_{MV.vision}.{ext}'
    else:
        p = (Path(outdir) if outdir else PRES_DIR) / f'mouse_view_f{f}_{MV.vision}.{ext}'
    p.parent.mkdir(parents=True, exist_ok=True)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB); h, w = img.shape[:2]
    if p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff', ''):
        cv2.imwrite(str(p), img)
    else:                                          # .pdf / .svg -> embed the raster view (container)
        fig = plt.figure(figsize=(w / 100, h / 100), dpi=100); axm = fig.add_axes([0, 0, 1, 1])
        axm.imshow(rgb); axm.axis('off'); fig.savefig(str(p), bbox_inches='tight', pad_inches=0); plt.close(fig)
    print('saved', p)
    if show:
        plt.figure(figsize=(9, 6.8)); plt.imshow(rgb); plt.axis('off')
        plt.title(f'{MOUSE_ID}  mouse view  frame {f}  ({MV.vision})'); plt.show()
    return p

mouse_view_png(F)                    # add vision='mouse' / fmt='pdf' / save_path='...' to redirect

## 4 — World-space map: where everything is

The bird's-eye complement to the mouse view — the whole trial in world coordinates: the avatar's
**path**, every **icon** (filled = on screen at this frame, open = off screen), the **avatar** and its
heading at the frame, and the **viewport box** (what the mouse can see). Saves like above.

In [ ]:
def world_map(frame, trial=None, outdir=None, save_path=None, fmt=None, show=True):
    f = int(frame); tr = MV.trial_of[f] if trial is None else trial
    a, b = trial_window(tr) if tr >= 0 else (max(0, f - 300), f + 300)
    ax, ay, th, ics = visible_icons(f)
    ts = MV.frame_ms[a:b]
    px = np.interp(ts, MV.COORD[:, 0], MV.COORD[:, 1]); py = np.interp(ts, MV.COORD[:, 0], MV.COORD[:, 2])
    fig, axm = plt.subplots(figsize=(7.6, 7.6))
    axm.add_patch(plt.Rectangle((0, 0), MV.W, MV.H, fill=False, ec='0.5', lw=1))          # world edge
    axm.scatter(px, py, c=np.arange(len(px)), cmap='plasma', s=6, alpha=0.7)              # avatar path
    vw, vh = MV.view_w, MV.view_h                    # viewport AT THIS FRAME -- it travels with him
    axm.add_patch(plt.Rectangle((ax - vw / 2, ay - vh / 2), vw, vh, facecolor='k', alpha=0.05,
                                edgecolor='k', lw=1.4, ls='--',
                                label=f'viewport @ this frame ({vw:.0f}x{vh:.0f} wu)'))
    mpl = {'reward': '#2ecc40', 'banish': '#3457d5', 'escape': '#17a2b8'}
    for i in ics:
        c = mpl.get(i['cls'], 'grey')
        axm.scatter([i['x']], [i['y']], s=320, marker='*',
                    facecolor=c if i['on'] else 'none', edgecolor=c, linewidths=1.8,
                    label=f"{i['cls']} ({'on' if i['on'] else 'off'})")
    axm.scatter([ax], [ay], s=140, c='k', marker='o', zorder=5, label='avatar')
    axm.arrow(ax, ay, 180 * np.cos(th), 180 * np.sin(th), head_width=55, color='k', zorder=5)
    axm.set_xlim(-80, MV.W + 80); axm.set_ylim(MV.H + 80, -80)      # y inverted = screen convention
    axm.set_aspect('equal'); axm.set_title(f'{MOUSE_ID}  trial {tr}  frame {f}  ({MV.world_of[f]})')
    h, l = axm.get_legend_handles_labels(); seen = dict(zip(l, h))
    axm.legend(seen.values(), seen.keys(), fontsize=7, loc='upper left', framealpha=0.9)
    fig.text(0.5, 0.005, 'dashed box = what he sees at THIS frame (the viewport moves with the avatar); '
             'coloured dots = the WHOLE trial path, so it naturally extends beyond one frame view',
             ha='center', fontsize=7.5, color='0.45')
    ext = (fmt or 'png').lstrip('.')
    if save_path:
        p = Path(save_path)
        if not p.suffix: p = p / f'world_map_f{f}.{ext}'
    else:
        p = (Path(outdir) if outdir else PRES_DIR) / f'world_map_f{f}.{ext}'
    p.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(p), dpi=130, bbox_inches='tight'); print('saved', p)
    plt.show() if show else plt.close(fig)
    return p

world_map(F)